# 18. 硬核降维打击：原生 CUDA C++ 编程与 PyTorch C++ 扩展 (JIT)

**难度：** Hard | **标签：** `CUDA C++`, `JIT Extension`, `Vector Add` | **目标人群：**核心 Infra 与算子开发

",
如果你在面试中只懂写 Triton，虽然能解决日常的算子融合问题，但面试官一定会问你：“Triton 底层的 Grid/Block 是怎么映射到 CUDA 线程的？”
本节我们将打破 Python 的温室，使用 `torch.utils.cpp_extension.load_inline`，直接在 Jupyter Notebook 的字符串里**手撕原生 CUDA C++ 核函数**，并且在后台即时 (JIT) 编译成 PyTorch 可用的模块。
我们将从最经典的 Vector Add (向量加法) 开始，直观对比 Triton 的 `pid` 和 CUDA 的 `blockIdx.x * blockDim.x + threadIdx.x`。


### Step 1: CUDA 编程模型核心

> **线程层级 (Thread Hierarchy)：**
> - **Grid (网格)**：整个计算任务的总称。由多个 Block 组成。
> - **Block (线程块)**：由多个 Thread 组成。同一个 Block 内的线程可以共享 SRAM (Shared Memory) 并进行同步。
> - **Thread (线程)**：GPU 计算的最小单位。
> 
> **计算全局索引的公式 (必考！)：**
> 如果我们把数据分为多个 Block，每个 Block 里有 `blockDim.x` 个线程。当前线程在整个大数组中的一维索引是：
> `int index = blockIdx.x * blockDim.x + threadIdx.x;`

> **与 Triton 的区别：**
> - 在 **Triton** 中，我们通常让一个 Program (相当于 CUDA 的一个 Block) 负责处理一个长度为 `BLOCK_SIZE` 的张量切片 (通过循环和 Mask)。
> - 在 **CUDA** 中，我们是为每一个标量元素 (Element) 分配一个 Thread！这种**细粒度**带来了极高的开发门槛，但也赋予了对硬件绝对的控制权。


### Step 2: CUDA 编程模型核心
从纯 Python 跨越到底层，最重要的是理解硬件线程层级：
1. **Thread (线程)**：底层的计算单元，负责单个数据元素。
2. **Block (线程块)**：一组协作的 Thread，共享极速小内存 (Shared Memory)，可在块内同步步调。通常包含 128 或 256 个线程。
3. **Grid (网格)**：一堆独立的 Block 组成 Grid，彼此无法直接通信。这极度贴合了矩阵并行的需求。


### Step 3: 原生 CUDA 与 PyTorch JIT 扩展框架
用 `__global__` 修饰 C++ CUDA Kernel，利用 `threadIdx.x` 和 `blockIdx.x` 定位数组下标。使用 PyTorch 的 `torch.utils.cpp_extension.load_inline`，它能在 Jupyter 运行时，唤起 `nvcc` 将 C++ 字符串编译成 Python 可直接引用的模块。


###  Step 4: 动手实战

**要求**：请补全下方 `cuda_source` 字符串中的 C++ CUDA 内核代码。你需要实现全局索引计算和越界检查。


In [ ]:
import torch
from torch.utils.cpp_extension import load_inline

# ==========================================
# 1. 编写原生 CUDA C++ 代码
# 我们定义一个 __global__ 关键字修饰的核函数
# 以及一个 C++ 的包裹函数用于给 PyTorch 调用
# ==========================================
cuda_source = '''
#include <torch/extension.h>
#include <cuda_runtime.h>

// CUDA 核函数 (Kernel)
__global__ void vector_add_kernel(const float* x, const float* y, float* out, int size) {
    // ==========================================
    // TODO 1: 计算当前线程的全局一维索引
    // ==========================================
    // int index = ???;
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    
    // ==========================================
    // TODO 2: 越界检查 (防止访问超过 size 的内存)
    // ==========================================
    // if (???) {
    if (index < size) {
        out[index] = x[index] + y[index];
    }
}

// PyTorch 调用的 C++ Wrapper 函数
torch::Tensor vector_add_cuda(torch::Tensor x, torch::Tensor y) {
    auto size = x.numel();
    auto out = torch::empty_like(x);
    
    // 设置线程块大小 (每个 Block 包含 256 个线程)
    const int threads = 256;
    
    // 计算需要的 Grid 大小 (向上取整)
    const int blocks = (size + threads - 1) / threads;
    
    // 调用核函数: kernel<<<blocks, threads>>>(...)
    vector_add_kernel<<<blocks, threads>>>(
        x.data_ptr<float>(), 
        y.data_ptr<float>(), 
        out.data_ptr<float>(), 
        size
    );
    
    // 确保 CUDA 执行完成 (同步)
    // 通常在 wrapper 里我们不需要手动 sync，因为 PyTorch 会管理流，但这里为了确保没报错加上
    cudaDeviceSynchronize();
    
    return out;
}
'''

# 用于向 PyTorch 注册我们写的 C++ 函数
cpp_source = '''
torch::Tensor vector_add_cuda(torch::Tensor x, torch::Tensor y);
'''

# ==========================================
# 2. 编译并加载 CUDA 扩展 (JIT)
# 这一步可能会耗时 10-30 秒，它会在后台调用 nvcc 编译器
# ==========================================
print("⏳ 正在后台使用 NVCC 编译 CUDA C++ 代码... 请耐心等待！")
import time
start = time.time()

try:
    vector_add_extension = load_inline(
        name='vector_add_ext',
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=['vector_add_cuda'],
        with_cuda=True,
        extra_cflags=['-O3'],
        extra_cuda_cflags=['-O3']
    )
    print(f"✅ 编译成功！耗时: {time.time() - start:.2f} 秒")
except Exception as e:
    print(f"❌ 编译失败，请确保环境中已安装 CUDA Toolkit (nvcc)。报错: {e}")


In [ ]:
# 测试你的 CUDA C++ 算子
def test_cuda_vector_add():
    if not torch.cuda.is_available() or 'vector_add_extension' not in globals():
        print("⏭️ 忽略测试：无 GPU 或编译失败。")
        return
        
    try:
        torch.manual_seed(42)
        # 故意选一个不能被 256 整除的 size，测试边界检查是否正确
        size = 10000 
        
        x = torch.randn(size, device='cuda', dtype=torch.float32)
        y = torch.randn(size, device='cuda', dtype=torch.float32)
        
        # 1. PyTorch 原生计算
        out_pt = x + y
        
        # 2. 我们手写的 CUDA C++ 计算
        out_cu = vector_add_extension.vector_add_cuda(x, y)
        
        diff = torch.max(torch.abs(out_pt - out_cu))
        assert diff < 1e-5, "CUDA 核函数计算结果错误！"
        
        print("✅ 太硬核了！你成功使用 JIT 编译并运行了原生的 CUDA C++ 代码！")
        print("💡 面试官如果问你 blockIdx.x 和 threadIdx.x，现在你可以把这个公式倒背如流了。")
        
    except Exception as e:
        print(f"❌ 测试失败: {e}")

test_cuda_vector_add()
